<a href="https://colab.research.google.com/github/ChanggonSong/AI-Study/blob/main/%EC%8B%A4%EC%8A%B52_1_%EC%8B%AC%EC%9E%A5%EC%A7%88%ED%99%98_%EB%B6%84%EB%A5%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 분류: 심장병 진단 유무 판별

In [314]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 커널을 구성하다보면 에러는 아니지만, 빨간색 네모 박스 warning이 뜨는 경우를 제거함
import warnings
warnings.filterwarnings('ignore')

In [315]:
# notebook을 실행한 브라우저에서 바로 그림을 볼 수 있게 해주는 라인
%matplotlib inline
# os 패키지를 통해 현재 디렉토리 위치를 변경하고, read_csv를 더 편리하게 함
import os
os.getcwd() # 현재 디렉토리 파악
os.chdir(r"/content/drive/MyDrive/Colab Notebooks/") # 불러오고 싶은 파일이 위치한 주소를 ___에 입력

In [316]:
# 다른 노트북 작성할 때도 이 셀만 떼서 사용 가능하다.
import matplotlib.pyplot as plt
import platform

# 웬만하면 해주는 것이 좋다.
from matplotlib import font_manager, rc
plt.rcParams['axes.unicode_minus']= False

if platform.system() == 'Darwin': # 맥os 사용자의 경우에
    plt.style.use('seaborn-darkgrid')
    rc('font', family = 'AppleGothic')

elif platform.system() == 'Windows':# 윈도우 사용자의 경우에
    path = 'c:/Windows/Fonts/malgun.ttf'
    font_name = font_manager.FontProperties(fname=path).get_name()
    plt.style.use('seaborn-darkgrid') # https://python-graph-gallery.com/199-matplotlib-style-sheets/
    rc('font', family=font_name)

In [317]:
# 한글이 들어간 csv는 encoding 인자를 넣어주는 것이 좋음
df = pd.read_csv('dataset.csv')
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [318]:
# 데이터 shape 파악
df.shape

(303, 14)

In [319]:
# 데이터 통계량 파악
df.describe()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000
mean,54.366337,0.683168,0.966997,131.623762,246.264026,0.148515,0.528053,149.646865,0.326733,1.039604,1.399340,0.729373,2.313531,0.544554
std,9.082101,0.466011,1.032052,17.538143,51.830751,0.356198,0.525860,22.905161,0.469794,1.161075,0.616226,1.022606,0.612277,0.498835
min,29.000000,0.000000,0.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,47.500000,0.000000,0.000000,120.000000,211.000000,0.000000,0.000000,133.500000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000
50%,55.000000,1.000000,1.000000,130.000000,240.000000,0.000000,1.000000,153.000000,0.000000,0.800000,1.000000,0.000000,2.000000,1.000000
75%,61.000000,1.000000,2.000000,140.000000,274.500000,0.000000,1.000000,166.000000,1.000000,1.600000,2.000000,1.000000,3.000000,1.000000
max,77.000000,1.000000,3.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,2.000000,4.000000,3.000000,1.000000


In [320]:
# 결측치 개수 파악
# 셀 실행 결과를 데이터프레임으로 보고 싶을 때 to_frame()과 pd.DataFrame() 두 가지를 사용 가능
df.isnull().sum().to_frame('nan_count')

,nan_count
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [321]:
# 결측치 비율 파악
pd.DataFrame(data=df.isnull().sum()/len(df),columns=['nan_ratio'])

,nan_ratio
age,0.0
sex,0.0
cp,0.0
trestbps,0.0
chol,0.0
fbs,0.0
restecg,0.0
thalach,0.0
exang,0.0
oldpeak,0.0


결측치가 있다면 결측치에 대한 전처리를 거쳐줘야 한다.

In [322]:
df.iloc[:,13].value_counts() # target에 대한 빈도수

1    165
0    138
Name: target, dtype: int64

In [323]:
y = df['target']
y

0      1
1      1
2      1
3      1
4      1
      ..
298    0
299    0
300    0
301    0
302    0
Name: target, Length: 303, dtype: int64

In [324]:
X=df.drop('target',axis=1)
X.head() # 불러온 데이터의 상위 5개를 출력

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2


In [325]:
patient_input = X.to_numpy()

In [326]:
patient_target = y.to_numpy()

In [327]:
# train & test
# 훈련세트와 테스트세트로 나누기
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(patient_input, patient_target, test_size=0.2, stratify=patient_target, random_state=42)

In [328]:
train_input.shape, test_input.shape, train_target.shape, test_target.shape

((242, 13), (61, 13), (242,), (61,))

In [329]:
# 검증세트 만들기
sub_input, val_input, sub_target, val_target = train_test_split(train_input, train_target, test_size=0.2, random_state=42)

In [330]:
print(sub_input.shape, val_input.shape)

(193, 13) (49, 13)


In [331]:
print(test_target)

[0 0 0 0 0 0 1 0 1 0 1 1 0 1 1 1 1 1 1 1 0 0 1 0 1 0 0 1 0 1 0 1 1 0 0 1 0
 1 1 1 0 1 1 1 0 0 1 0 1 0 0 1 1 0 1 1 1 1 0 0 1]


In [332]:
# test_target의 심장병 진단 비율
count_0 = np.sum(test_target == 0)
count_1 = np.sum(test_target == 1)
print("Count of 0:", count_0)
print("Count of 1:", count_1)
print(count_1 / count_0)

# 원래 심장병 진단 비율
value_counts_result = df.iloc[:,13].value_counts()
count_1 = value_counts_result.get(1, 0)
count_0 = value_counts_result.get(0, 0)
print("Count of 0:", count_0)
print("Count of 1:", count_1)
print(count_1 / count_0)

Count of 0: 28
Count of 1: 33
1.1785714285714286
Count of 0: 138
Count of 1: 165
1.1956521739130435


테스트 세트의 비율이 맞다. (샘플링 편향 문제 해결)

In [333]:
# 표준화 전처리 - StandardScaler
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
ss.fit(train_input)
train_scaled = ss.transform(train_input)
test_scaled = ss.transform(test_input)

In [334]:
# SVM

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC

clf_svm = SVC(random_state=0)
clf_svm.fit(train_scaled, train_target)

pred_svm = clf_svm.predict(test_scaled)

print("\n--- SVM Classifier ---")
print(accuracy_score(test_target, pred_svm))
print(confusion_matrix(test_target, pred_svm))


--- SVM Classifier ---
0.819672131147541
[[19  9]
 [ 2 31]]


In [335]:
# LR

from sklearn.linear_model import LogisticRegression

clf_lr = LogisticRegression(random_state=0)
clf_lr.fit(train_scaled, train_target)

pred_lr = clf_lr.predict(test_scaled)

print ("\n--- Logistic Regression Classifier ---")
print (accuracy_score(test_target, pred_lr))
print (confusion_matrix(test_target, pred_lr))


--- Logistic Regression Classifier ---
0.8032786885245902
[[19  9]
 [ 3 30]]


In [336]:
# DT

from sklearn.tree import DecisionTreeClassifier

clf_dt = DecisionTreeClassifier(random_state=0)
#clf_dt.fit(train_scaled, train_target)
clf_dt.fit(sub_input, sub_target)

pred_dt = clf_dt.predict(val_input)

print ("\n--- Decision Tree Classifier ---")
print (accuracy_score(val_target, pred_dt))
print (confusion_matrix(val_target, pred_dt))


--- Decision Tree Classifier ---
0.7755102040816326
[[ 9  6]
 [ 5 29]]


In [338]:
# RT

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

print ("\n--- Random Forest ---")
rf_clf = RandomForestClassifier(random_state=0)
rf_clf.fit(train_scaled, train_target)
pred = rf_clf.predict(test_scaled)
print(accuracy_score(test_target,pred))
print (confusion_matrix(test_target, pred))


--- Random Forest ---
0.8032786885245902
[[19  9]
 [ 3 30]]
